# RQ1, part 2 — one label set per sentence (one-vs-rest)

**Why a second notebook.** `eligibility_classifier.ipynb` gives every sentence ONE label. In production that lost requirements: the call sentence

> "Applicants must be over 18 years of age, be proficient in French or English, and reside in South Africa, Benin, … or Senegal."

was labeled `AGE` (0.66; `RESIDENCE` came second at 0.27), so the engine only checked the age and never the residence. A non-resident artist could be told nothing rules them out. The annotation rounds had hidden the problem: a sentence with two independent requirements was split by hand into parts (`…_0a`, `…_0b`), so the model was trained on one requirement per chunk, while the product classifies whole sentences.

**Task.** Predict the SET of requirement labels a sentence states: any of the 10 requirement classes (`RESIDENCE`, `NATIONALITY`, `AGE`, `DISCIPLINE`, `CAREER_STAGE`, `EDUCATION`, `STUDENT_STATUS`, `APPLICANT_TYPE`, `PRIOR_FUNDING`, `OTHER_ELIGIBILITY`). `NONE` is the empty set.

**Data.** `dataset/labels/eligibility_annotations_multilabel.csv`, built by `scripts/build_multilabel_dataset.py` (round 4, `ANNOTATION_GUIDELINES.md` §7): the round 1–3 labels, the hand-split sentences rejoined into the whole sentence the product sees, second labels where a sentence states another requirement, and a batch of compound-looking corpus sentences.

**Approach.** One-vs-rest: one binary classifier per label, each answering "does this sentence state a requirement of this type?". Labels no longer compete for probability mass, so a sentence can carry two. Same models as part 1, same validation (5-fold `StratifiedGroupKFold` by opportunity, stratified on each sentence's first label, 10 fold seeds), so results stay comparable.

**Metrics.** Multi-label metrics replace accuracy: micro-F1 and macro-F1 over the 10 requirement labels, exact-set accuracy (subset accuracy), Hamming loss, plus two product metrics:
- **missed requirements**: share of true (sentence, label) pairs the model does not predict. A miss can turn into a wrong "nothing rules you out", so this is the number the engine cares about most;
- **compound recall**: the same, on multi-label sentences only, which is the case this notebook exists for.

In [1]:
import os
import sys

import numpy as np
import pandas as pd

REPO_ROOT = os.path.dirname(os.getcwd())
sys.path.insert(0, REPO_ROOT)
DATA_PATH = os.path.join(REPO_ROOT, "dataset", "labels", "eligibility_annotations_multilabel.csv")

df = pd.read_csv(DATA_PATH)
df["label_set"] = df["labels"].str.split("|").apply(lambda labels: [l for l in labels if l != "NONE"])
df["primary"] = df["labels"].str.split("|").str[0]   # for stratifying the folds only
print(df.shape)
print(df["source"].value_counts().to_dict())
print("labels per sentence:", df["label_set"].str.len().value_counts().sort_index().to_dict())

(692, 8)
{'single': 506, 'compound': 75, 'rejoined': 56, 'relabel': 55}
labels per sentence: {0: 222, 1: 325, 2: 109, 3: 23, 4: 12, 5: 1}


In [2]:
from sklearn.preprocessing import MultiLabelBinarizer

REQUIREMENT_LABELS = ["RESIDENCE", "NATIONALITY", "AGE", "DISCIPLINE", "CAREER_STAGE", "EDUCATION",
                      "STUDENT_STATUS", "APPLICANT_TYPE", "PRIOR_FUNDING", "OTHER_ELIGIBILITY"]
mlb = MultiLabelBinarizer(classes=REQUIREMENT_LABELS)
Y = mlb.fit_transform(df["label_set"])          # one 0/1 column per requirement label; NONE = all zeros
groups = df["opportunity_id"]
compound = Y.sum(axis=1) >= 2

pd.DataFrame({"sentences": Y.sum(axis=0), "in compound sentences": Y[compound].sum(axis=0)},
             index=REQUIREMENT_LABELS)

,sentences,in compound sentences
RESIDENCE,108,79
NATIONALITY,40,29
AGE,42,34
DISCIPLINE,129,68
CAREER_STAGE,97,59
EDUCATION,18,9
STUDENT_STATUS,41,5
APPLICANT_TYPE,113,39
PRIOR_FUNDING,31,4
OTHER_ELIGIBILITY,46,14


In [3]:
from sklearn.model_selection import StratifiedGroupKFold


def folds_for(seed):
    # StratifiedGroupKFold can't stratify on a label matrix, so it stratifies on each sentence's
    # first label (what part 1 stratified on) and still keeps every call on one side of the split
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    return list(cv.split(df, df["primary"], groups))


folds = folds_for(42)
for fold, (train_idx, test_idx) in enumerate(folds):
    assert not set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]), "group leakage"
    print(f"fold {fold}: train={len(train_idx)}  test={len(test_idx)}  compound in test={compound[test_idx].sum()}")

fold 0: train=553  test=139  compound in test=32
fold 1: train=553  test=139  compound in test=35
fold 2: train=554  test=138  compound in test=26
fold 3: train=554  test=138  compound in test=25
fold 4: train=554  test=138  compound in test=27


In [4]:
from sklearn.metrics import accuracy_score, f1_score, hamming_loss, recall_score


def scores(Y_true, Y_pred):
    # every metric over the 10 requirement labels; an all-zero row is a correct NONE
    return {
        "micro_f1": f1_score(Y_true, Y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(Y_true, Y_pred, average="macro", zero_division=0),
        "subset_acc": accuracy_score(Y_true, Y_pred),            # whole label set exactly right
        "hamming": hamming_loss(Y_true, Y_pred),                 # share of wrong 0/1 cells
        "missed": 1 - recall_score(Y_true, Y_pred, average="micro", zero_division=0),
        "compound_recall": recall_score(Y_true[Y_true.sum(axis=1) >= 2], Y_pred[Y_true.sum(axis=1) >= 2],
                                        average="micro", zero_division=0),
    }


def fold_mean(Y_pred, fold_list):
    # mean of the per-fold scores, like GridSearchCV's best_score_ in part 1
    return pd.DataFrame([scores(Y[t], Y_pred[t]) for _, t in fold_list]).mean()

### Model 0: the current single-label model, scored as a label set

Part 1's chosen model (bge-base embeddings + Logistic Regression, C=100), trained on each sentence's first label and predicting one label, read as a set of one (or the empty set for `NONE`). This is the reference: what the product did until now, measured on the same sentences and folds.

In [5]:
from fastembed import TextEmbedding

EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"   # the product's embedding model (part 1, "Model 3")
CACHE = os.path.join(REPO_ROOT, "notebooks", "cache", "emb_bge-base_multilabel.npz")

cached = np.load(CACHE, allow_pickle=True) if os.path.exists(CACHE) else None
if cached is not None and list(cached["chunk_text"]) == df["chunk_text"].tolist():
    X_emb = cached["X"]
else:  # sentences changed (or no cache): embed again. Each sentence is embedded on its own, so
       # embedding everything up front leaks nothing between folds (part 1, cell 12)
    embedder = TextEmbedding(EMBEDDING_MODEL, cache_dir=os.path.expanduser("~/.cache/fastembed"))
    X_emb = np.array(list(embedder.embed(df["chunk_text"].tolist(), batch_size=8)), dtype=np.float32)
    np.savez_compressed(CACHE, chunk_text=df["chunk_text"].to_numpy(), X=X_emb)
X_emb.shape

/Users/lucatomarelli/personal_projects/open_art/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(692, 768)

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

single = LogisticRegression(C=100, class_weight="balanced", max_iter=5000)   # part 1's final configuration


def single_label_as_set(fold_list):
    predicted = cross_val_predict(single, X_emb, df["primary"], cv=fold_list, n_jobs=-1)
    return mlb.transform([[] if label == "NONE" else [label] for label in predicted])


Y_single = single_label_as_set(folds)
fold_mean(Y_single, folds).round(3)

micro_f1           0.605
macro_f1           0.585
subset_acc         0.544
hamming            0.066
missed             0.475
compound_recall    0.385
dtype: float64

### Models 1–3: one-vs-rest

Same three product candidates as part 1, each wrapped in `OneVsRestClassifier` (one binary model per label, `class_weight="balanced"` inside each, since every label is rare against the rest). Hyperparameters by `GridSearchCV` on the same folds, scored on macro-F1 over the 10 labels.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

X_text = df[["chunk_text"]]

candidates = {
    "TF-IDF + LogReg (baseline)": (
        Pipeline([("features", ColumnTransformer([("tfidf", TfidfVectorizer(), "chunk_text")])),
                  ("clf", OneVsRestClassifier(LogisticRegression(class_weight="balanced", max_iter=2000)))]),
        {"features__tfidf__ngram_range": [(1, 1), (1, 2)], "features__tfidf__min_df": [1, 2, 3],
         "clf__estimator__C": [0.1, 1, 10, 100]},
        X_text),
    "TF-IDF + Linear SVM": (
        Pipeline([("features", ColumnTransformer([("tfidf", TfidfVectorizer(), "chunk_text")])),
                  ("clf", OneVsRestClassifier(LinearSVC(class_weight="balanced", max_iter=10000)))]),
        {"features__tfidf__ngram_range": [(1, 1), (1, 2)], "features__tfidf__min_df": [1, 2, 3],
         "clf__estimator__C": [0.01, 0.1, 1, 10]},
        X_text),
    "bge-base + LogReg": (
        Pipeline([("clf", OneVsRestClassifier(LogisticRegression(class_weight="balanced", max_iter=5000)))]),
        {"clf__estimator__C": [0.01, 0.1, 1, 10, 100, 1000]},
        X_emb),
}

searches = {}
for name, (pipeline, grid, X_all) in candidates.items():
    search = GridSearchCV(pipeline, grid, cv=folds, scoring="f1_macro", n_jobs=-1)
    search.fit(X_all, Y, groups=groups)
    searches[name] = search
    print(f"{name:28s} best macro-F1 {search.best_score_:.3f}  {search.best_params_}")

TF-IDF + LogReg (baseline)   best macro-F1 0.609  {'clf__estimator__C': 1, 'features__tfidf__min_df': 2, 'features__tfidf__ngram_range': (1, 1)}


TF-IDF + Linear SVM          best macro-F1 0.609  {'clf__estimator__C': 0.1, 'features__tfidf__min_df': 3, 'features__tfidf__ngram_range': (1, 1)}


bge-base + LogReg            best macro-F1 0.676  {'clf__estimator__C': 100}


### Per-label thresholds (bge-base + LogReg)

A binary model says "yes" at probability 0.5 by default. With `class_weight="balanced"` that is already tilted towards recall, but one cut-off doesn't fit ten labels with 18 to 129 examples each. Each label's threshold is tuned to its best F1, **nested**: inside every outer training fold an inner 5-fold CV gives out-of-fold probabilities, the thresholds are picked on those, and only then applied to the outer test fold. The test fold never influences its own thresholds. On a tie the lower threshold wins: a missed requirement costs more than an extra check.

In [8]:
from sklearn.base import clone

THRESHOLD_GRID = np.round(np.arange(0.05, 0.96, 0.05), 2)
emb_best = searches["bge-base + LogReg"].best_estimator_


def pick_thresholds(probabilities, Y_true):
    # per label, the threshold with the best F1 on these (out-of-fold) probabilities; lowest on a tie
    chosen = []
    for j in range(Y_true.shape[1]):
        f1s = [f1_score(Y_true[:, j], probabilities[:, j] >= t, zero_division=0) for t in THRESHOLD_GRID]
        chosen.append(float(THRESHOLD_GRID[int(np.argmax(f1s))]))
    return np.array(chosen)


def tuned_predictions(estimator, X_all, fold_list):
    # nested: thresholds chosen inside each training fold, applied to its test fold
    Y_pred = np.zeros_like(Y)
    for train_idx, test_idx in fold_list:
        inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
        inner_folds = list(inner.split(train_idx, df["primary"].iloc[train_idx], groups.iloc[train_idx]))
        inner_proba = cross_val_predict(clone(estimator), X_all[train_idx], Y[train_idx], cv=inner_folds,
                                        method="predict_proba", n_jobs=-1)
        thresholds = pick_thresholds(inner_proba, Y[train_idx])
        model = clone(estimator).fit(X_all[train_idx], Y[train_idx])
        Y_pred[test_idx] = model.predict_proba(X_all[test_idx]) >= thresholds
    return Y_pred


Y_emb_default = cross_val_predict(emb_best, X_emb, Y, cv=folds, n_jobs=-1)
Y_emb_tuned = tuned_predictions(emb_best, X_emb, folds)
pd.DataFrame({"threshold 0.5": fold_mean(Y_emb_default, folds),
              "tuned per label (nested)": fold_mean(Y_emb_tuned, folds)}).round(3)

,threshold 0.5,tuned per label (nested)
micro_f1,0.706,0.691
macro_f1,0.676,0.670
subset_acc,0.575,0.564
hamming,0.057,0.060
missed,0.282,0.303
compound_recall,0.742,0.720


### Robust comparison: 10 fold seeds

As in part 1, one split can flatter a model, so every model is re-run over 10 fold seeds with the hyperparameters its seed-42 search picked (the same small optimistic bias for all). Reported as mean ± std.

In [9]:
SEEDS = range(10)
rows = []
for seed in SEEDS:
    seed_folds = folds_for(seed)
    predictions = {"single-label bge-base (current)": single_label_as_set(seed_folds)}
    for name, (_, _, X_all) in candidates.items():
        predictions[f"OvR {name}"] = cross_val_predict(searches[name].best_estimator_, X_all, Y, cv=seed_folds, n_jobs=-1)
    predictions["OvR bge-base + LogReg, tuned thresholds"] = tuned_predictions(emb_best, X_emb, seed_folds)
    for name, Y_pred in predictions.items():
        rows.append({"model": name, "seed": seed, **fold_mean(Y_pred, seed_folds)})

robust = pd.DataFrame(rows)
summary = robust.drop(columns="seed").groupby("model", sort=False).agg(["mean", "std"])
summary.round(3)

micro_f1        macro_f1         \
                                            mean    std     mean    std   
model                                                                     
single-label bge-base (current)            0.595  0.005    0.572  0.006   
OvR TF-IDF + LogReg (baseline)             0.649  0.007    0.610  0.013   
OvR TF-IDF + Linear SVM                    0.640  0.008    0.609  0.012   
OvR bge-base + LogReg                      0.715  0.005    0.681  0.007   
OvR bge-base + LogReg, tuned thresholds    0.704  0.007    0.671  0.013   

                                        subset_acc        hamming         \
                                              mean    std    mean    std   
model                                                                      
single-label bge-base (current)              0.544  0.009   0.067  0.001   
OvR TF-IDF + LogReg (baseline)               0.529  0.009   0.066  0.001   
OvR TF-IDF + Linear SVM                      0.502  0.011   0.071  0.002   
OvR bge-base + LogReg                        0.589  0.008   0.056  0.001   
OvR bge-base + LogReg, tuned thresholds      0.577  0.010   0.058  0.002   

                                        missed        compound_recall         
                                          mean    std            mean    std  
model                                                                         
single-label bge-base (current)          0.490  0.004           0.373  0.009  
OvR TF-IDF + LogReg (baseline)           0.359  0.007           0.676  0.008  
OvR TF-IDF + Linear SVM                  0.343  0.010           0.688  0.015  
OvR bge-base + LogReg                    0.270  0.004           0.755  0.007  
OvR bge-base + LogReg, tuned thresholds  0.284  0.011           0.736  0.017

In [10]:
# per-label F1 and recall, seed 42, chosen model (threshold 0.5): where does one-vs-rest gain, where does it lose?
from sklearn.metrics import precision_recall_fscore_support

per_label = {}
for name, Y_pred in {"single-label (current)": Y_single, "OvR bge-base (chosen)": Y_emb_default}.items():
    p, r, f, _ = precision_recall_fscore_support(Y, Y_pred, zero_division=0)
    per_label[(name, "recall")] = r
    per_label[(name, "F1")] = f
pd.DataFrame(per_label, index=REQUIREMENT_LABELS).round(2)

single-label (current)       OvR bge-base (chosen)      
                                  recall    F1                recall    F1
RESIDENCE                           0.44  0.55                  0.84  0.82
NATIONALITY                         0.45  0.57                  0.78  0.75
AGE                                 0.64  0.75                  0.98  0.94
DISCIPLINE                          0.57  0.64                  0.74  0.71
CAREER_STAGE                        0.54  0.63                  0.73  0.73
EDUCATION                           0.28  0.40                  0.28  0.34
STUDENT_STATUS                      0.83  0.83                  0.83  0.81
APPLICANT_TYPE                      0.56  0.59                  0.65  0.62
PRIOR_FUNDING                       0.58  0.64                  0.61  0.62
OTHER_ELIGIBILITY                   0.20  0.25                  0.35  0.37

In [11]:
# the per-label confusion matrices (multi-label has one 2x2 matrix per label: tn fp / fn tp)
from sklearn.metrics import multilabel_confusion_matrix

cms = multilabel_confusion_matrix(Y, Y_emb_default)
pd.DataFrame([{"label": label, "tn": cm[0, 0], "fp": cm[0, 1], "fn": cm[1, 0], "tp": cm[1, 1]}
              for label, cm in zip(REQUIREMENT_LABELS, cms)]).set_index("label")

,tn,fp,fn,tp
label,,,,
RESIDENCE,562,22,17,91
NATIONALITY,640,12,9,31
AGE,646,4,1,41
DISCIPLINE,518,45,33,96
CAREER_STAGE,568,27,26,71
EDUCATION,668,6,13,5
STUDENT_STATUS,642,9,7,34
APPLICANT_TYPE,531,48,40,73
PRIOR_FUNDING,650,11,12,19


In [12]:
# errors on compound sentences, seed 42: which second requirements are still missed?
pd.set_option("display.max_colwidth", 140)
gold = df["label_set"].apply(set)
predicted = pd.Series([set(mlb.classes_[row.astype(bool)]) for row in Y_emb_default])
missed = (gold - predicted).where(compound)
errors = df.assign(missed=missed, predicted=predicted)[missed.apply(lambda s: bool(s) if isinstance(s, set) else False)]
print(f"{len(errors)} of {compound.sum()} compound sentences still miss at least one label")
errors[["chunk_text", "labels", "predicted", "missed"]].head(15)

62 of 145 compound sentences still miss at least one label


,chunk_text,labels,predicted,missed
0,"Funding is available for professional artists and cultural practitioners (such as curators, producers, literary translators, cultural ed...",DISCIPLINE|CAREER_STAGE,{DISCIPLINE},{CAREER_STAGE}
18,HBO level circus or dance trained,EDUCATION|DISCIPLINE,"{OTHER_ELIGIBILITY, DISCIPLINE}",{EDUCATION}
43,You are pursuing a master’s degree or have just graduated as a musician,STUDENT_STATUS|DISCIPLINE,{STUDENT_STATUS},{DISCIPLINE}
63,"Benelux Interest: Specifically tailored for organizations from Flanders, Luxembourg, the Netherlands, and Wallonia looking for high-impa...",RESIDENCE|APPLICANT_TYPE,{RESIDENCE},{APPLICANT_TYPE}
113,a program for particularly gifted young visual artists in contemporary art.,DISCIPLINE|CAREER_STAGE,{},"{DISCIPLINE, CAREER_STAGE}"
152,"Bulgarian private cultural organizations within the meaning of Article 10, paragraph 1 of the ZZRK;",RESIDENCE|APPLICANT_TYPE,{},"{RESIDENCE, APPLICANT_TYPE}"
187,Professional performing art groups.,CAREER_STAGE|DISCIPLINE|APPLICANT_TYPE,"{CAREER_STAGE, DISCIPLINE}",{APPLICANT_TYPE}
192,"Professional musicians, artists, ensembles and bands can apply for grants for concert and touring activities within all genres.",DISCIPLINE|CAREER_STAGE,{DISCIPLINE},{CAREER_STAGE}
271,"Professional, producing performing arts institutions can receive grants under this regulation.",DISCIPLINE|CAREER_STAGE|APPLICANT_TYPE,{DISCIPLINE},"{APPLICANT_TYPE, CAREER_STAGE}"
272,Grants are awarded to performing arts institutions that have staged new drama for an audience.,APPLICANT_TYPE|DISCIPLINE,{},"{APPLICANT_TYPE, DISCIPLINE}"


## Conclusion

*Snapshot on the round-4 dataset (692 sentences, 145 multi-label, 273 calls). Mean ± std over 10 fold seeds ("Robust comparison" cell).*

| Model | micro-F1 | macro-F1 (10 labels) | exact set | missed requirements | compound recall |
|---|---|---|---|---|---|
| Single-label bge-base + LogReg (until now) | 0.595 ± 0.005 | 0.572 ± 0.006 | 0.544 | 49.0% | 0.373 |
| OvR TF-IDF + LogReg (baseline) | 0.649 ± 0.007 | 0.610 ± 0.013 | 0.529 | 35.9% | 0.676 |
| OvR TF-IDF + Linear SVM | 0.640 ± 0.008 | 0.609 ± 0.012 | 0.502 | 34.3% | 0.688 |
| **OvR bge-base + LogReg, threshold 0.5 — chosen** | **0.715 ± 0.005** | **0.681 ± 0.007** | **0.589** | **27.0%** | **0.755** |
| OvR bge-base + LogReg, per-label thresholds (nested) | 0.704 ± 0.007 | 0.671 ± 0.013 | 0.577 | 28.4% | 0.736 |

**Answer to RQ1, part 2: yes.** A sentence can state several requirements, and a one-vs-rest model finds them: compared with the single-label model on the same sentences and folds, it misses **27% instead of 49%** of the requirements stated, and on compound sentences it recovers **76% instead of 37%** of their labels. It is ahead on every metric, by margins many times the seed-to-seed spread, including exact-set accuracy, where a single label should do well on the 79% of sentences that have at most one.

**Chosen model: one-vs-rest bge-base + Logistic Regression, C=100, threshold 0.5 for every label.**
- The same representation and classifier as part 1: only the decision changes from "the most probable class" to "every label whose own model says yes".
- Embeddings beat TF-IDF here too, by ~0.07 macro-F1, the same gap as in part 1.
- **Per-label thresholds were tested and rejected:** tuned honestly (nested), they are worse on every metric. With ~550 training sentences per fold the tuned cut-offs are noise, and `class_weight="balanced"` already pushes each binary model towards recall at 0.5.

**Where it is strong, where it is not** (per-label table, seed 42): the requirements the engine can reject on are found well: `AGE` 0.94 F1 (recall 0.98), `RESIDENCE` 0.82 (recall 0.84, up from 0.44), `STUDENT_STATUS` 0.81, `NATIONALITY` 0.75 (recall 0.78, up from 0.45). The weak labels are the same as in part 1: `OTHER_ELIGIBILITY` (0.37, a class defined by exclusion) and `EDUCATION` (0.34, recall 0.28 as before: only 18 sentences, the one label more data would clearly help). Most remaining compound misses are second labels hidden in a fused noun phrase ("**professional** musicians", "performing arts **institutions**"), which the engine treats as check-only anyway (`CAREER_STAGE`, `DISCIPLINE`) or which the safety net and the reviews still guard (`APPLICANT_TYPE`).

**Limits.**
- Macro-F1 here is not comparable with part 1's 0.701: it is a different task (a set per sentence, not one class) on a different dataset (whole sentences instead of split ones), averaged over 10 requirement labels instead of 11 classes.
- The round-4 labels were made by a Claude Code session following the guidelines; unlike rounds 2–3 they have no human review yet.
- The compound batch was picked because it looked compound, so compound sentences are over-represented (21% of the dataset; 15.7% of the corpus sentences get two or more labels from the shipped model, `scripts/extract_constraints.py`).

## Final model

The chosen configuration is refit on **all** sentences, since nothing needs to be held back any more. A label is predicted at probability ≥ 0.5 (no per-label tuning, see the conclusion). Its expected quality is the 10-seed CV estimate above.

Saved next to part 1's model, which stays as the single-label reference:
- `artifacts/eligibility_classifier_multilabel.joblib`: the fitted one-vs-rest Logistic Regression (embeddings in, one probability per requirement label out)
- `artifacts/eligibility_classifier_multilabel.json`: the embedding model it expects, its labels and **thresholds**, the CV scores and the library versions

In [13]:
import json
from datetime import date

import fastembed
import joblib
import sklearn

final_model = clone(emb_best).fit(X_emb, Y)
final_thresholds = np.full(len(REQUIREMENT_LABELS), 0.5)   # tuned thresholds lost to 0.5 (conclusion)

ARTIFACT_DIR = os.path.join(REPO_ROOT, "artifacts")
MODEL_PATH = os.path.join(ARTIFACT_DIR, "eligibility_classifier_multilabel.joblib")
META_PATH = os.path.join(ARTIFACT_DIR, "eligibility_classifier_multilabel.json")
joblib.dump(final_model, MODEL_PATH)

chosen = "OvR bge-base + LogReg"
metadata = {
    "task": "RQ1 multi-label eligibility classification of requirements_text_en sentences",
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dim": int(X_emb.shape[1]),
    "classifier": "OneVsRestClassifier(LogisticRegression)",
    "params": {k.removeprefix("clf__estimator__"): v for k, v in searches["bge-base + LogReg"].best_params_.items()},
    "class_weight": "balanced",
    "labels": REQUIREMENT_LABELS,              # column order of predict_proba; no label above its threshold = NONE
    "thresholds": dict(zip(REQUIREMENT_LABELS, final_thresholds.tolist())),
    "n_train_chunks": int(len(df)),
    "n_multilabel_chunks": int(compound.sum()),
    "n_train_opportunities": int(groups.nunique()),
    "training_data": "dataset/labels/eligibility_annotations_multilabel.csv",
    "cv": "5-fold StratifiedGroupKFold by opportunity_id (stratified on the first label), 10 seeds",
    "cv_10_seeds": {metric: {"mean": round(float(summary.loc[chosen, (metric, "mean")]), 4),
                             "std": round(float(summary.loc[chosen, (metric, "std")]), 4)}
                    for metric in ["micro_f1", "macro_f1", "subset_acc", "missed", "compound_recall"]},
    "trained_on": date.today().isoformat(),
    "versions": {"scikit-learn": sklearn.__version__, "fastembed": fastembed.__version__},
}
with open(META_PATH, "w") as f:
    json.dump(metadata, f, indent=2)
metadata["thresholds"]

{'RESIDENCE': 0.5,
 'NATIONALITY': 0.5,
 'AGE': 0.5,
 'DISCIPLINE': 0.5,
 'CAREER_STAGE': 0.5,
 'EDUCATION': 0.5,
 'STUDENT_STATUS': 0.5,
 'APPLICANT_TYPE': 0.5,
 'PRIOR_FUNDING': 0.5,
 'OTHER_ELIGIBILITY': 0.5}

In [14]:
# Sanity check: reload from disk (as the product will) and label unseen sentences, including the
# production sentence that motivated this notebook. joblib files are pickles: only ever load this one.
loaded = joblib.load(MODEL_PATH)
assert (loaded.predict_proba(X_emb) == final_model.predict_proba(X_emb)).all()

embedder = TextEmbedding(EMBEDDING_MODEL, cache_dir=os.path.expanduser("~/.cache/fastembed"))
examples = [
    "Applicants must be resident in Scotland.",
    "The residency is open to artists under the age of 35.",
    "Applicants must be over 18 years of age , be proficient in French or English, and reside in South Africa, "
    "Benin, Cameroon, Ghana, Guinea, Kenya, Mali, Mozambique, Uganda, the Republic of the Congo, Rwanda or Senegal.",
    "The selected artist will present their work at the end of the residency.",
]
proba = loaded.predict_proba(np.array(list(embedder.embed(examples))))
pd.DataFrame({
    "sentence": [e[:80] for e in examples],
    "labels": [[l for l, p, t in zip(REQUIREMENT_LABELS, row, final_thresholds) if p >= t] or ["NONE"] for row in proba],
    "probabilities": [{l: round(float(p), 2) for l, p in zip(REQUIREMENT_LABELS, row) if p >= 0.1} for row in proba],
})

,sentence,labels,probabilities
0,Applicants must be resident in Scotland.,[RESIDENCE],{'RESIDENCE': 0.99}
1,The residency is open to artists under the age of 35.,[AGE],"{'RESIDENCE': 0.19, 'AGE': 1.0, 'OTHER_ELIGIBILITY': 0.43}"
2,"Applicants must be over 18 years of age , be proficient in French or English, an","[RESIDENCE, AGE, OTHER_ELIGIBILITY]","{'RESIDENCE': 0.96, 'NATIONALITY': 0.16, 'AGE': 0.99, 'OTHER_ELIGIBILITY': 0.95}"
3,The selected artist will present their work at the end of the residency.,[OTHER_ELIGIBILITY],"{'CAREER_STAGE': 0.18, 'OTHER_ELIGIBILITY': 0.76}"
